# Calibration Importance Analysis

Analyzes data from the **Calibration Importance Test** experiment.

**Research question:** Does using your own calibration vs. someone else's calibration affect head-tracking performance?

Upload 2 files:
- Participant A's calibration data (`calibration-test-*A*.csv`)
- Participant B's calibration data (`calibration-test-*B*.csv`)

All metrics (Wₑ, IDₑ, TP) are computed using the **ISO 9241-9 directional projection** method.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (12, 6)
matplotlib.rcParams['font.size'] = 11
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
# Upload your CSV files — change filenames as needed
FILE_A = 'calibration-test-Roberto-2026-02-17T22_48_45.271Z.csv'  # <-- change if needed
FILE_B = 'calibration-test-Soha-2026-02-17T22_52_09.816Z.csv'     # <-- change if needed

LABEL_A = 'Roberto\'s Calibration'  # <-- label for plots
LABEL_B = 'Soha\'s Calibration'     # <-- label for plots

import os
def load_csv(filename):
    paths = [
        filename,
        os.path.join(os.path.expanduser('~/Downloads'), filename),
    ]
    for p in paths:
        if os.path.exists(p):
            print(f'Loaded: {p}')
            return pd.read_csv(p)
    raise FileNotFoundError(f'Could not find {filename}')

data_a = load_csv(FILE_A)
data_b = load_csv(FILE_B)

data_a['calibration'] = LABEL_A
data_b['calibration'] = LABEL_B

print(f'{LABEL_A}: {len(data_a)} trials')
print(f'  Source: {data_a["calibrationSource"].iloc[0]}')
print(f'{LABEL_B}: {len(data_b)} trials')
print(f'  Source: {data_b["calibrationSource"].iloc[0]}')

## 1. Recompute Metrics with Corrected Wₑ

Using ISO 9241-9 directional projection (combined-coordinate method).

In [ ]:
def compute_fitts_metrics(df, label):
    """Compute ISO 9241-9 metrics using directional projection for Wₑ."""
    results = []
    
    for layout_idx in sorted(df['layoutIndex'].unique()):
        layout = df[(df['layoutIndex'] == layout_idx) & 
                    (df['movementTime'].notna()) & 
                    (df['movementTime'] > 0)]
        
        if len(layout) < 2:
            continue
        
        mean_mt = layout['movementTime'].mean()
        Ae = layout['actualAmplitude'].mean()
        
        theta_rad = np.radians(layout['direction'].astype(float))
        dx = layout['selectionX'] - layout['targetX']
        dy = layout['selectionY'] - layout['targetY']
        projections = dx * np.cos(theta_rad) + dy * np.sin(theta_rad)
        SDx = projections.std(ddof=1)
        We = 4.133 * SDx
        
        IDe = np.log2(Ae / We + 1) if We > 0 else 0
        TP = IDe / mean_mt if mean_mt > 0 else 0
        
        target_size = layout['targetSize'].iloc[0]
        amplitude = layout['amplitude'].iloc[0]
        
        # Error rate
        errors = layout['selectionError'] > (target_size / 2)
        error_rate = errors.sum() / len(layout)
        
        results.append({
            'calibration': label,
            'layoutIndex': layout_idx,
            'targetSize': target_size,
            'amplitude': amplitude,
            'N': len(layout),
            'MeanMT': mean_mt,
            'Ae': Ae,
            'We': We,
            'IDe': IDe,
            'TP': TP,
            'MeanError': layout['selectionError'].mean(),
            'ErrorRate': error_rate
        })
    
    return pd.DataFrame(results)

results_a = compute_fitts_metrics(data_a, LABEL_A)
results_b = compute_fitts_metrics(data_b, LABEL_B)
results = pd.concat([results_a, results_b], ignore_index=True)

print(f'Computed metrics: {len(results_a)} conditions for {LABEL_A}, {len(results_b)} for {LABEL_B}')
display(results.round(4))

## 2. Overall Comparison

In [ ]:
overall = results.groupby('calibration').agg(
    AvgTP=('TP', 'mean'),
    StdTP=('TP', 'std'),
    AvgMT=('MeanMT', 'mean'),
    StdMT=('MeanMT', 'std'),
    AvgWe=('We', 'mean'),
    AvgIDe=('IDe', 'mean'),
    AvgError=('MeanError', 'mean'),
    AvgErrorRate=('ErrorRate', 'mean')
).reset_index()

print('=== Overall Comparison ===')
print(f'{"Calibration":>25} {"TP (bps)":>14} {"MT (s)":>14} {"We (px)":>10} {"Error (px)":>12} {"Error%":>8}')
print('-' * 87)
for _, row in overall.iterrows():
    print(f'{row["calibration"]:>25} {row["AvgTP"]:>5.3f}±{row["StdTP"]:.3f} '
          f'{row["AvgMT"]:>5.3f}±{row["StdMT"]:.3f} '
          f'{row["AvgWe"]:>10.2f} {row["AvgError"]:>12.2f} {row["AvgErrorRate"]*100:>7.1f}%')

tp_a = overall[overall['calibration'] == LABEL_A]['AvgTP'].values[0]
tp_b = overall[overall['calibration'] == LABEL_B]['AvgTP'].values[0]
better = LABEL_A if tp_a > tp_b else LABEL_B
diff_pct = abs(tp_a - tp_b) / min(tp_a, tp_b) * 100
print(f'\n{better} has {diff_pct:.1f}% higher throughput.')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colors = {'A': '#3b82f6', 'B': '#f97316'}

# Throughput
ax = axes[0]
for i, (_, row) in enumerate(overall.iterrows()):
    ax.bar(i, row['AvgTP'], yerr=row['StdTP'], capsize=5,
           color=list(colors.values())[i], alpha=0.8)
ax.set_xticks(range(len(overall)))
ax.set_xticklabels(overall['calibration'], rotation=15, ha='right')
ax.set_ylabel('Throughput (bits/s)')
ax.set_title('Throughput')

# Movement Time
ax = axes[1]
for i, (_, row) in enumerate(overall.iterrows()):
    ax.bar(i, row['AvgMT'], yerr=row['StdMT'], capsize=5,
           color=list(colors.values())[i], alpha=0.8)
ax.set_xticks(range(len(overall)))
ax.set_xticklabels(overall['calibration'], rotation=15, ha='right')
ax.set_ylabel('Movement Time (s)')
ax.set_title('Movement Time')

# Selection Error
ax = axes[2]
for i, (_, row) in enumerate(overall.iterrows()):
    ax.bar(i, row['AvgError'],
           color=list(colors.values())[i], alpha=0.8)
ax.set_xticks(range(len(overall)))
ax.set_xticklabels(overall['calibration'], rotation=15, ha='right')
ax.set_ylabel('Selection Error (px)')
ax.set_title('Average Selection Error')

plt.suptitle('Calibration Comparison: Overall Performance', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 3. Per-Layout Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Throughput by layout
ax = axes[0]
layouts = sorted(results['layoutIndex'].unique())
x = np.arange(len(layouts))
width = 0.35

tp_a_vals = [results_a[results_a['layoutIndex']==l]['TP'].values[0] if len(results_a[results_a['layoutIndex']==l]) > 0 else 0 for l in layouts]
tp_b_vals = [results_b[results_b['layoutIndex']==l]['TP'].values[0] if len(results_b[results_b['layoutIndex']==l]) > 0 else 0 for l in layouts]

ax.bar(x - width/2, tp_a_vals, width, label=LABEL_A, color='#3b82f6')
ax.bar(x + width/2, tp_b_vals, width, label=LABEL_B, color='#f97316')
ax.set_xlabel('Layout')
ax.set_ylabel('Throughput (bits/s)')
ax.set_title('Throughput by Layout')

# Build layout labels
layout_labels = []
for l in layouts:
    row = results[results['layoutIndex'] == l].iloc[0]
    layout_labels.append(f'{row["targetSize"]:.0f}px\n{row["amplitude"]:.0f}px')
ax.set_xticks(x)
ax.set_xticklabels(layout_labels, fontsize=9)
ax.legend()

# Movement Time by layout
ax = axes[1]
mt_a_vals = [results_a[results_a['layoutIndex']==l]['MeanMT'].values[0] if len(results_a[results_a['layoutIndex']==l]) > 0 else 0 for l in layouts]
mt_b_vals = [results_b[results_b['layoutIndex']==l]['MeanMT'].values[0] if len(results_b[results_b['layoutIndex']==l]) > 0 else 0 for l in layouts]

ax.bar(x - width/2, mt_a_vals, width, label=LABEL_A, color='#3b82f6')
ax.bar(x + width/2, mt_b_vals, width, label=LABEL_B, color='#f97316')
ax.set_xlabel('Layout')
ax.set_ylabel('Movement Time (s)')
ax.set_title('Movement Time by Layout')
ax.set_xticks(x)
ax.set_xticklabels(layout_labels, fontsize=9)
ax.legend()

plt.tight_layout()
plt.show()

## 4. Fitts' Law Regression

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 6))

for label, color, marker, res in [(LABEL_A, '#3b82f6', 'o', results_a), 
                                   (LABEL_B, '#f97316', 's', results_b)]:
    ax.scatter(res['IDe'], res['MeanMT'], color=color, marker=marker, s=60, alpha=0.7, label=label)
    
    if len(res) > 1:
        coeffs = np.polyfit(res['IDe'], res['MeanMT'], 1)
        x_fit = np.linspace(res['IDe'].min(), res['IDe'].max(), 100)
        r2 = np.corrcoef(res['IDe'], res['MeanMT'])[0,1]**2
        ax.plot(x_fit, np.polyval(coeffs, x_fit), '--', color=color, alpha=0.8,
                label=f'{label}: MT={coeffs[1]:.2f}+{coeffs[0]:.2f}·IDe (R²={r2:.3f})')

ax.set_xlabel('Effective Index of Difficulty IDₑ (bits)')
ax.set_ylabel('Movement Time (s)')
ax.set_title('Fitts\' Law: MT vs. IDₑ')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 5. Raw Movement Time Distributions

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 5))

mt_a = data_a['movementTime'].dropna()
mt_b = data_b['movementTime'].dropna()

ax.hist(mt_a, bins=20, alpha=0.6, color='#3b82f6', 
        label=f'{LABEL_A} (M={mt_a.mean():.2f}s, SD={mt_a.std():.2f}s)')
ax.hist(mt_b, bins=20, alpha=0.6, color='#f97316', 
        label=f'{LABEL_B} (M={mt_b.mean():.2f}s, SD={mt_b.std():.2f}s)')
ax.axvline(mt_a.mean(), color='#3b82f6', linestyle='--', linewidth=2)
ax.axvline(mt_b.mean(), color='#f97316', linestyle='--', linewidth=2)
ax.set_xlabel('Movement Time (s)')
ax.set_ylabel('Count')
ax.set_title('Movement Time Distribution')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Summary Table

In [ ]:
print('=== Per-Layout Detailed Comparison ===')
print(f'{"Layout":>6} {"Size":>6} {"Amp":>6} | '
      f'{LABEL_A[:10]+"..":>12} TP  {"MT":>6} | '
      f'{LABEL_B[:10]+"..":>12} TP  {"MT":>6} | {"TP Diff":>8}')
print('-' * 85)

for layout_idx in sorted(results['layoutIndex'].unique()):
    ra = results_a[results_a['layoutIndex'] == layout_idx]
    rb = results_b[results_b['layoutIndex'] == layout_idx]
    if len(ra) == 0 or len(rb) == 0:
        continue
    ra = ra.iloc[0]
    rb = rb.iloc[0]
    tp_diff = ra['TP'] - rb['TP']
    sign = '+' if tp_diff > 0 else ''
    print(f'{int(layout_idx):>6} {ra["targetSize"]:>6.0f} {ra["amplitude"]:>6.0f} | '
          f'{ra["TP"]:>12.3f} {ra["MeanMT"]:>6.3f} | '
          f'{rb["TP"]:>12.3f} {rb["MeanMT"]:>6.3f} | {sign}{tp_diff:>7.3f}')

print(f'\nPositive TP Diff = {LABEL_A} is better')
print(f'Negative TP Diff = {LABEL_B} is better')